# Sentiment Analysis MyTelkomsel v2 (satu notebook)

**Cara pakai:** jalankan berurutan dari atas. Setelah selesai, pakai *Restart kernel and Run All* untuk memastikan semuanya konsisten.

**Aturan notebook ini:** semua logika ditulis sebagai **fungsi** (sel bertanda `# -> src/...`). Nanti kalau ingin dipecah, cukup salin fungsi-fungsi itu ke file `.py`; sel lain (yang memanggil fungsi) tetap di notebook.

| Bagian | Isi |
|---|---|
| 0 | Setup & konfigurasi |
| 1 | Ambil data (scrape) |
| 2 | Preprocessing (negasi & emoji dipertahankan) |
| 3 | Label dari rating + diagnosa konflik teks vs rating |
| 4 | Gold set (label manual untuk evaluasi) |
| 5 | Baseline TF-IDF + Logistic Regression |
| 6 | Analisis error |
| 7 | IndoBERT: ekspor data ke Colab + nilai hasilnya |
| 8 | Coba prediksi |
| 9 | Evaluasi akhir di gold_test (sekali) |

**Kenapa begini:** di notebook awal, label dibuat dari lexicon sehingga model hanya meniru lexicon (tidak peka sarkasme). Di sini model dilatih dari rating (weak label) dan **dinilai pada label manual**.

## 0. Setup & konfigurasi

In [1]:
# Jalankan sekali jika library belum ada (hapus tanda # di baris berikut):
# !pip install pandas numpy scikit-learn joblib google-play-scraper

In [2]:
import re, json, time, joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# ---------- KONFIGURASI ----------
APP_ID      = "com.telkomsel.telkomselcm"   # MyTelkomsel
N_REVIEWS   = 12000
SORT        = "newest"       # "newest" lebih representatif daripada "relevant"
RUN_SCRAPE  = False          # True = ambil ulang data dari Play Store (butuh internet)
SEED        = 42

DATA_DIR  = Path("data");  DATA_DIR.mkdir(exist_ok=True)
RAW_CSV   = DATA_DIR / "reviews_raw.csv"
GOLD_CSV  = DATA_DIR / "gold_final.csv"            # satu-satunya file gold yang dipakai evaluasi
GOLD_SPLIT_CSV = DATA_DIR / "gold_split.csv"      # pembagian TETAP gold -> val / test (jangan diedit manual)
GOLD_BLIND   = DATA_DIR / "1_gold_blind_100.csv"   # dilabeli manusia lebih dulu, tanpa melihat rating/label LLM
GOLD_REVIEW  = DATA_DIR / "2_gold_review_400.csv"  # lalu Anda periksa baris PERIKSA dan AUDIT
CLAUDE_PRELABEL = DATA_DIR / "3_llm_prelabel_500.csv"   # pra-label LLM (Claude); dipakai untuk kalibrasi
GOLD_RELABEL = DATA_DIR / "4_relabel_campuran.csv"  # label ulang ulasan campuran (aturan: keluhan konkret -> negative)
DROP_NOISY_POSITIVES = True   # buang dari TRAINING review bintang 4-5 yang isinya keluhan (lihat bagian 5)
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
BERT_DIR  = DATA_DIR / "bert"                      # pertukaran file dengan notebook Colab IndoBERT
RUN_FINAL_TEST = False   # True HANYA sekali di akhir (bagian 9), setelah semua keputusan dibuat di gold_val

# 2 kelas: "positive" = BUKAN keluhan (puas, netral tanpa keluhan, pertanyaan/saran tanpa nada negatif).
# Review bintang 3 tidak punya label rating yang bisa dipercaya -> dikeluarkan dari data latih.
TWO_CLASS = True
LABELS   = ["negative", "positive"] if TWO_CLASS else ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

## 1. Ambil data
Notebook awal memakai `MOST_RELEVANT` (condong ke review panjang/ber-upvote; hasilnya 82% negatif). Default di sini `newest`.
Jika `data/reviews_raw.csv` sudah ada, scraping dilewati. Untuk mengambil ulang, set `RUN_SCRAPE = True` dan hapus file lamanya.

In [3]:
# -> src/scrape.py
def scrape_reviews(count=N_REVIEWS, sort=SORT, app_id=APP_ID):
    from google_play_scraper import Sort, reviews
    sorts = {"newest": Sort.NEWEST, "relevant": Sort.MOST_RELEVANT, "rating": Sort.RATING}
    rows, token = [], None
    while len(rows) < count:
        batch, token = reviews(app_id, lang="id", country="id", sort=sorts[sort],
                               count=min(200, count - len(rows)), continuation_token=token)
        if not batch:
            break
        rows.extend(batch)
        print(f"  {len(rows)}/{count}")
        if token is None:
            break
        time.sleep(0.5)
    return pd.DataFrame(rows)

In [4]:
if RUN_SCRAPE:
    fresh = scrape_reviews()
    if RAW_CSV.exists():   # GABUNGKAN dengan data lama; jangan menimpa (gold set bergantung pada reviewId lama)
        fresh = pd.concat([pd.read_csv(RAW_CSV), fresh]).drop_duplicates("reviewId", keep="first")
    fresh.to_csv(RAW_CSV, index=False)
    print(f"Raw sekarang {len(fresh)} review (data lama dipertahankan).")

if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"{RAW_CSV} belum ada. Set RUN_SCRAPE=True untuk mengambil data, "
        "atau salin CSV hasil scraping lama Anda ke path tersebut (kolom minimal: reviewId, content, score).")

raw = pd.read_csv(RAW_CSV)
print(raw.shape)
raw[["content", "score"]].head()

(12000, 11)


,content,score
0,"lelet,Kouta mahal boros adeh",1
1,trimakasih,5
2,pulsa sering hilang sendiri padahal paket masi...,1
3,"Internet LAMBAT BANGET B*NGS*T, AKU GAK MAU LI...",1
4,sangat bagus,5


## 2. Preprocessing
Perbedaan dari notebook awal: **negasi dipertahankan** (`tidak`, `bukan`, `belum`, ...) dan digabung jadi `tidak_bagus`; **emoji dan `!`/`?` dipertahankan**; tanpa stopword NLTK penuh (isinya bisa membuang negasi); tanpa stemming; kamus slang diaudit.

Dua keluaran: `text_light` (untuk transformer & char n-gram) dan `text_classic` (untuk TF-IDF kata).

In [5]:
# -> resources/slang_id.json  (kamus slang yang sudah diaudit; entri ambigu 1-2 huruf dibuang)
SLANG = json.loads(r'''{"7an": "tujuan", "ababil": "abg labil", "abis": "habis", "acc": "accord", "adlah": "adalah", "adlh": "adalah", "adoh": "aduh", "afaik": "as far as i know", "aha": "tertawa", "ahaha": "haha", "ahokncc": "ahok", "aing": "saya", "aj": "saja", "aja": "saja", "ajj": "saja", "aka": "dikenal juga sebagai", "akhirx": "akhirnya", "akika": "aku", "akko": "aku", "akkoh": "aku", "akku": "aku", "akuwh": "aku", "akyu": "aku", "alay": "norak", "aljasa": "asal jadi saja", "alow": "halo", "ama": "sama", "ambilin": "ambilkan", "ambl": "ambil", "analisise": "analisis", "ancur": "hancur", "anjir": "anjing", "anjrit": "anjing", "ank": "anak", "anter": "antar", "ap2": "apa-apa", "apaan": "apa", "apasih": "apa sih", "ape": "apa", "aplot": "unggah", "aps": "apa", "apva": "apa", "aqu": "aku", "aquwh": "aku", "asap": "sesegera mungkin", "asbun": "asal bunyi", "aseek": "asyik", "aseekk": "asyik", "asek": "asyik", "asekk": "asyik", "aseknya": "asyiknya", "asoy": "asyik", "astrojim": "astagfirullahaladzim", "astul": "asal tulis", "ath": "kalau begitu", "ato": "atau", "atuh": "kalau begitu", "ava": "avatar", "awak": "saya", "aws": "awas", "ayang": "sayang", "ayank": "sayang", "ayok": "ayo", "bacot": "banyak bicara", "bagusss": "bagus", "bakalan": "akan", "bales": "balas", "bandes": "bantuan desa", "bangdes": "pembangunan desa", "bangedh": "banget", "bangkotan": "tua", "banpol": "bantuan polisi", "banpres": "bantuan presiden", "banpur": "bantuan tempur", "bansarkas": "bantuan sarana kesehatan", "basbang": "basi", "bazis": "badan amal, zakat, infak, dan sedekah", "bcanda": "bercanda", "bcoz": "karena", "bdg": "bandung", "beb": "sayang", "begajulan": "nakal", "bejibun": "banyak", "beliin": "belikan", "belom": "belum", "benarjujur": "jujur", "bener": "benar", "bentar": "sebentar", "ber2": "berdua", "ber3": "bertiga", "berdikari": "berdiri di atas kaki sendiri", "beresin": "membereskan", "bet": "banget", "beti": "beda tipis", "beud": "banget", "beut": "banget", "bgd": "banget", "bgmn": "bagaimana", "bgs": "bagus", "bgt": "banget", "bgtt": "banget", "bgttt": "banget", "bgus": "bagus", "bhubu": "tidur", "bijimane": "bagaimana", "bimbuluh": "bimbingan dan penyuluhan", "bintal": "bimbingan mental", "bisi": "kalau-kalau", "bkl": "akan", "bkn": "bukan", "bknnya": "bukannya", "blegug": "bodoh", "blg": "bilang", "blh": "boleh", "blm": "belum", "bln": "bulan", "bls": "balas", "blum": "belum", "bnchi": "benci", "bnci": "benci", "bngung": "bingung", "bnran": "yang benar", "bnyk": "banyak", "bodor": "lucu", "bohay": "badan aduhai", "bokep": "porno", "bokin": "pacar", "bokis": "bohong", "bole": "boleh", "boljug": "boleh juga", "bolot": "bodoh", "bonek": "bocah nekat", "bonyok": "ayah ibu", "boyeh": "boleh", "bpk": "bapak", "brb": "segera kembali", "brg": "bareng", "brngkt": "berangkat", "brp": "berapa", "bru": "baru", "brur": "saudara laki-laki", "bs": "bisa", "bsa": "bisa", "bsen": "bosan", "bsk": "besok", "btw": "ngomong-ngomong", "bu_bu": "tidur", "buangbuang": "buang", "bubarin": "bubarkan", "bubbu": "tidur", "buber": "buka bersama", "bubu": "tidur", "bujubune": "luar biasa", "bumil": "ibu hamil", "buser": "buru sergap", "bwhn": "bawahan", "bwt": "buat", "byar": "bayar", "byk": "banyak", "byr": "bayar", "byrin": "bayarkan", "cabut": "pergi", "cadas": "keren", "caem": "cakep", "calo": "makelar", "cama-cama": "sama-sama", "cangcut": "celana dalam", "cape": "capek", "caper": "cari perhatian", "caur": "jelek", "cekak": "tidak ada uang", "cekal": "cegah tangkal", "cekidot": "coba lihat", "cemplungin": "cemplungkan", "cengengesan": "tertawa", "ceper": "pendek", "cepet": "cepat", "ceu": "kakak perempuan", "cew": "cewek", "cewe": "cewek", "chuyunk": "sayang", "cibuk": "sibuk", "cimeng": "ganja", "cin": "cinta", "ciye": "cie", "ciyh": "sih", "ckck": "ck", "ckepp": "cakep", "ckp": "cakep", "clbk": "cinta lama bersemi kembali", "cmiiw": "correct me if i'm wrong", "cmpr": "campur", "cmpur": "campur", "cnenk": "senang", "cod": "bayar ditempat", "congor": "mulut", "cow": "cowok", "cowwyy": "maaf", "coz": "karena", "cpa": "siapa", "cpe": "capek", "cppe": "capek", "cucok": "cocok", "cuex": "cuek", "cups": "culun", "curanmor": "pencurian kendaraan bermotor", "cwek": "cewek", "cyin": "cinta", "dah": "deh", "dapet": "dapat", "dek": "adik", "deyh": "deh", "dgn": "dengan", "diancurin": "dihancurkan", "dimaafin": "dimaafkan", "dimintak": "diminta", "disono": "di sana", "dket": "dekat", "dkk": "dan kawan-kawan", "dll": "dan lain-lain", "dlu": "dulu", "dngn": "dengan", "dodol": "bodoh", "doku": "uang", "dongs": "dong", "dpt": "dapat", "dr": "dari", "dri": "dari", "drmn": "darimana", "drtd": "dari tadi", "dst": "dan seterusnya", "dtg": "datang", "duh": "aduh", "duren": "durian", "egp": "emang gue pikirin", "eke": "aku", "elu": "kamu", "emangnya": "memangnya", "emng": "memang", "endak": "tidak", "enggak": "tidak", "envy": "iri", "eror": "error", "error": "error", "faktakta": "fakta", "fax": "facsimile", "fifo": "first in first out", "folbek": "follow back", "fyi": "sebagai informasi", "ga": "tidak", "gaada": "tidak ada uang", "gag": "tidak", "gaje": "tidak jelas", "gak": "tidak", "gakbisa": "tidak bisa", "gan": "juragan", "gantengsaya": "ganteng", "gaptek": "gagap teknologi", "gatek": "gagap teknologi", "gawe": "kerja", "gbs": "tidak bisa", "gebetan": "orang yang disuka", "geje": "tidak jelas", "gepeng": "gelandangan dan pengemis", "ghiy": "lagi", "gile": "gila", "gimana": "bagaimana", "gino": "gigi nongol", "githu": "gitu", "gk": "tidak", "gmana": "bagaimana", "goblok": "bodoh", "gokil": "gila", "golput": "golongan putih", "gombal": "suka merayu", "gowes": "mengayuh sepeda", "gpl": "tidak pakai lama", "gpny": "tidak punya", "gpp": "tidak apa-apa", "gretong": "gratis", "gretongan": "gratisan", "gtau": "tidak tahu", "gtw": "tidak tahu", "gua": "saya", "gue": "saya", "guoblok": "goblok", "guys": "teman-teman", "gw": "saya", "gws": "cepat sembuh", "haghaghag": "tertawa", "haha": "tertawa", "hakhak": "tertawa", "hallow": "halo", "handak": "bahan peledak", "hankam": "pertahanan dan keamanan", "hansip": "pertahanan sipil", "hardcore": "keras", "hehe": "he", "hellow": "halo", "helo": "halo", "helow": "halo", "heters": "pembenci", "hey": "hai", "hlm": "halaman", "hlng": "hilang", "hny": "hanya", "hnya": "hanya", "hoax": "isu bohong", "houm": "rumah", "hrs": "harus", "hrus": "harus", "hubad": "hubungan angkatan darat", "hubdar": "perhubungan darat", "hubla": "perhubungan laut", "huff": "mengeluh", "huft": "mengeluh", "hum": "rumah", "humas": "hubungan masyarakat", "humz": "rumah", "idk": "saya tidak tahu", "ikon": "lambang", "ilang": "hilang", "ilfeel": "tidak suka", "ilfil": "tidak suka", "imba": "jago sekali", "imho": "in my humble opinion", "imoet": "imut", "imoetz": "imut", "indo": "indonesia", "info": "informasi", "isengin": "bercanda", "ista": "nista", "istaa": "nista", "item": "hitam", "itung": "hitung", "itungan": "hitungan", "iyala": "iya lah", "iye": "iya", "iyo": "iya", "jablay": "jarang dibelai", "jadiin": "jadi", "jadul": "jaman dulu", "jaim": "jaga image", "jancuk": "anjing", "jayus": "tidak lucu", "jd": "jadi", "jdi": "jadi", "jdikan": "jadikan", "jelekk": "jelek", "jem": "jam", "jentelmen": "berani", "jg": "juga", "jga": "juga", "jgn": "jangan", "jgnkan": "jangankan", "jijay": "jijik", "jir": "anjing", "jkt": "jakarta", "jlk": "jelek", "jln": "jalan", "jnj": "janji", "jomblo": "tidak punya pacar", "jth": "jatuh", "jubir": "juru bicara", "jurdil": "jujur adil", "jutek": "galak", "jwb": "jawab", "kab": "kabupaten", "kabag": "kepala bagian", "kabor": "kabur", "kacian": "kasihan", "kacrut": "kacau", "kadit": "kepala direktorat", "kadiv": "kepala divisi", "kaga": "tidak", "kagak": "tidak", "kaka": "kakak", "kalo": "kalau", "kampret": "sialan", "kamtib": "keamanan dan ketertiban", "kamtibmas": "keamanan dan ketertiban masyarakat", "kamuh": "kamu", "kamuwh": "kamu", "kamyu": "kamu", "kanwil": "kantor wilayah", "kapt": "kapten", "karna": "karena", "kasat": "kepala satuan", "kasubbag": "kepala subbagian", "kasubbid": "kepala subbidang", "katrok": "kampungan", "kau": "kamu", "kayanya": "kayaknya", "kbar": "kabar", "kbr": "kabar", "kcian": "kasihan", "kdu": "harus", "keburu": "terlanjur", "kec": "kecamatan", "kedubes": "kedutaan besar", "kejurnas": "kejuaraan nasional", "kek": "seperti", "kekeuh": "keras kepala", "keknya": "kayaknya", "kel": "kelurahan", "keliatan": "kelihatan", "kemaren": "kemarin", "keneh": "masih", "kentot": "senggama", "kepengen": "mau", "kepikiran": "terpikirkan", "kepingin": "mau", "kepo": "mau tahu urusan orang", "kepsek": "kepala sekolah", "kere": "tidak punya uang", "kesbang": "kesatuan bangsa", "kesian": "kasihan", "kesra": "kesejahteraan rakyat", "ketauan": "ketahuan", "ketrima": "diterima", "keukeuh": "keras kepala", "kgiatan": "kegiatan", "khan": "kan", "kibul": "bohong", "kibus": "kaki busuk", "kimpoi": "kawin", "kl": "kalau", "klian": "kalian", "klianz": "kalian", "klo": "kalau", "kloter": "kelompok terbang", "kluarga": "keluarga", "klw": "kalau", "klwrga": "keluarga", "km": "kamu", "kmari": "kemari", "kmps": "kampus", "kmpus": "kampus", "kmrn": "kemarin", "knal": "kenal", "knl": "kenal", "knp": "kenapa", "knpa": "kenapa", "koar": "teriak", "kodya": "kota madya", "kog": "kok", "komdis": "komisi disiplin", "kompi": "komputer", "komsov": "komunis sovyet", "komtiong": "komunis Tiongkok", "kongkow": "kumpul bareng teman-teman", "konjen": "konsulat jenderal", "kopdar": "kopi darat", "koq": "kok", "korup": "korupsi", "kpd": "kepada", "kpn": "kapan", "kptsan": "keputusan", "krenz": "keren", "krik": "garing", "krm": "kirim", "krn": "karena", "ktauan": "ketahuan", "ktipu": "tipu", "ktmu": "ketemu", "ktny": "katanya", "ktr": "kantor", "kudu": "harus", "kuper": "kurang pergaulan", "kuq": "kok", "kurng": "kurang", "kyk": "seperti", "kykny": "kayanya", "lag": "lag", "laka": "kecelakaan", "lam": "salam", "lambreta": "lambat", "lamp": "lampiran", "lansia": "lanjut usia", "lanud": "landasan udara", "lapas": "lembaga pemasyarakatan", "latgab": "latihan gabungan", "lbur": "libur", "lebay": "berlebihan", "leh": "boleh", "lekong": "laki-laki", "lelet": "lambat", "lemot": "lambat", "lg": "lagi", "lgi": "lagi", "lgkp": "lengkap", "lgsg": "langsung", "lht": "lihat", "liat": "lihat", "light": "terang", "linmas": "perlindungan masyarakat", "litbang": "penelitian dan pengembangan", "lmyan": "lumayan", "lmyn": "lumayan", "lngkp": "lengkap", "loch": "loh", "loe": "kamu", "lol": "tertawa", "lola": "lambat berfikir", "lom": "belum", "louph": "cinta", "loupz": "cinta", "low": "kalau", "lowh": "kamu", "lu": "kamu", "luber": "langsung, umum, bebas, dan rahasia", "luchu": "lucu", "luchuw": "lucu", "luff": "cinta", "lum": "belum", "luph": "cinta", "luthu": "lucu", "lwn": "lawan", "lwt": "lewat", "maacih": "terima kasih", "maaciw": "terima kasih", "mabal": "bolos", "mabes": "markas besar", "macem": "macam", "macem-macem": "macam-macam", "macih": "masih", "madesu": "masa depan suram", "maem": "makan", "maen": "main", "magabut": "makan gaji buta", "mahatma": "maju sehat bersama", "maho": "homo", "mak": "ibu", "makasih": "terima kasih", "maks": "maksimal", "maksain": "memaksa", "malah": "bahkan", "malem": "malam", "malu2in": "memalukan", "mam": "makan", "mamz": "makan", "maneh": "kamu", "manies": "manis", "maniez": "manis", "mantab": "mantap", "mantep": "mantap", "mantul": "mantap", "mao": "mau", "markus": "makelar kasus", "masi": "masih", "masukin": "masukkan", "masy": "masyarakat", "mba": "mbak", "mbual": "bual", "melu": "ikut", "mending": "lebih baik", "mepet": "dekat sekali", "mgkin": "mungkin", "mgkn": "mungkin", "mgu": "minggu", "mhn": "mohon", "migas": "minyak dan gas bumi", "miker": "minuman keras", "mikol": "minuman beralkohol", "milis": "mailing list", "miras": "minuman keras", "mksd": "maksud", "mlah": "malah", "mls": "malas", "mngkn": "mungkin", "mnolak": "tolak", "mnt": "minta", "moge": "motor gede", "mokad": "mati", "mokat": "mati", "moso": "masa", "mosok": "masa", "most": "paling", "mpe": "sampai", "msh": "masih", "msk": "masuk", "mskpn": "meskipun", "mslh": "masalah", "msng2": "masing-masing", "muahal": "mahal", "mubes": "musyawarah besar", "muker": "musyawarah kerja", "mulu": "melulu", "mumet": "pusing", "mumpung": "selagi", "muna": "munafik", "munas": "musyawarah nasional", "munaslub": "musyawarah nasional luar biasa", "munfik": "munafik", "muntaber": "muntah dan berak", "musda": "musyawarah daerah", "musti": "mesti", "muup": "maaf", "muupz": "maaf", "muuv": "maaf", "nal": "kenal", "nanam": "menanam", "nangis": "menangis", "nanya": "bertanya", "naon": "apa", "napa": "kenapa", "napi": "narapidana", "napol": "narapidana politik", "napza": "narkotika, alkohol, psikotropika, dan zat adiktif ", "naq": "anak", "narkoba": "narkotika, psikotropika, dan obat terlarang", "narsis": "bangga pada diri sendiri", "nasgor": "nasi goreng", "nax": "anak", "nda": "tidak", "ndak": "tidak", "ndiri": "sendiri", "ndut": "gendut", "nekolim": "neokolonialisme", "nekolin": "neokolonialisme", "nelfon": "menelepon", "nembak": "menyatakan cinta", "next": "lanjut", "ngabis2in": "menghabiskan", "ngabuburit": "menunggu berbuka puasa", "ngakak": "tertawa", "ngaku": "mengaku", "ngambek": "marah", "ngambil": "mengambil", "ngampus": "pergi ke kampus", "nganggur": "tidak punya pekerjaan", "ngantri": "mengantri", "ngapah": "kenapa", "ngapain": "sedang apa", "ngaret": "terlambat", "ngaruh": "berpengaruh", "ngasih": "memberikan", "ngawur": "berbicara sembarangan", "ngebandel": "berbuat bandel", "ngebully": "intimidasi", "ngeceng": "kumpul bareng-bareng", "ngegosip": "bergosip", "ngeh": "sadar", "ngehe": "kesal", "ngeklaim": "mengklaim", "ngekos": "tinggal di kos", "ngeksis": "menjadi eksis", "ngelag": "lag", "ngelamar": "melamar", "ngeles": "berkilah", "ngeliat": "melihat", "ngelidur": "menggigau", "ngemeng": "bicara terus-terusan", "ngentotin": "senggama", "ngerampok": "merampok", "ngerti": "mengerti", "ngga": "tidak", "nggak": "tidak", "ngibul": "berbohong", "ngikut": "ikut", "ngiler": "mau", "nginep": "menginap", "ngiri": "iri", "ngisi": "mengisi", "ngisiin": "mengisikan", "ngmg": "bicara", "ngmng": "bicara", "ngocol": "lucu", "ngomong": "bicara", "ngomongin": "membicarakan", "ngubek2": "mencari-cari", "ngumpet": "sembunyi", "ngumpul": "berkumpul", "ngurus": "mengurus", "nie": "ini", "nih": "ini", "nipu": "tipu", "niyh": "nih", "nmenarik": "menarik", "nmr": "nomor", "nntn": "nonton", "nobar": "nonton bareng", "ntar": "nanti", "ntn": "nonton", "numpuk": "bertumpuk", "nutupin": "menutupi", "nyari": "mencari", "nyariin": "mencari", "nyasar": "tersesat", "nyekar": "menyekar", "nyela": "cela", "nyembah": "sembah", "nyerang": "serang", "nyiapin": "mempersiapkan", "nyicil": "mencicil", "nyiram": "menyiram", "nyoblos": "mencoblos", "nyok": "ayo", "nyokap": "ibu", "ogah": "tidak mau", "ongkir": "ongkos kirim", "oot": "out of topic", "org2": "orang-orang", "ortu": "orang tua", "otda": "otonomi daerah", "otw": "on the way, sedang di jalan", "pacal": "pacar", "pake": "pakai", "pala": "kepala", "pansus": "panitia khusus", "paranoid": "takut", "parpol": "partai politik", "parte": "partai", "pasutri": "pasangan suami istri", "pd": "pada", "pede": "percaya diri", "pelatnas": "pemusatan latihan nasional", "pemda": "pemerintah daerah", "pemkot": "pemerintah kota", "pemred": "pemimpin redaksi", "penertiban": "tertib", "penjas": "pendidikan jasmani", "perda": "peraturan daerah", "perhatiin": "perhatikan", "pesenan": "pesanan", "pgang": "pegang", "pilkada": "pemilihan kepala daerah", "pisan": "sangat", "plg": "paling", "plis": "tolong", "plisss": "tolong", "pls": "tolong", "pmrnth": "pemerintah", "pndukung": "pendukung", "polantas": "polisi lalu lintas", "ponpes": "pondok pesantren", "prg": "pergi", "priksa": "periksa", "prnh": "pernah", "pro": "profesional", "psen": "pesan", "psn": "pesan", "pst": "pasti", "psti": "pasti", "pswt": "pesawat", "puanas": "panas", "qmo": "kamu", "qmu": "kamu", "rakor": "rapat koordinasi", "rame": "ramai", "ranmor": "kendaraan bermotor", "raskin": "rakyat miskin", "red": "redaksi", "ref": "referensi", "reg": "register", "rehab": "rehabilitasi", "rejeki": "rezeki", "reklamuk": "reklamasi", "rempong": "sulit", "renstra": "rencana strategis", "repp": "balas", "reskrim": "reserse kriminal", "restik": "reserse narkotika", "rhs": "rahasia", "rights": "benar", "rmh": "rumah", "rubahnn": "rubah", "ruko": "rumah toko", "rusunawa": "rumah susun sewa", "ruz": "terus", "sadaar": "sadar", "saia": "saya", "salting": "salah tingkah", "sampe": "sampai", "samsek": "sama sekali", "sapose": "siapa", "satpam": "satuan pengamanan", "sbb": "sebagai berikut", "sbh": "sebuah", "sbnrny": "sebenarnya", "scr": "secara", "sdgkn": "sedangkan", "sdkt": "sedikit", "se7": "setuju", "sembako": "sembilan bahan pokok", "sempet": "sempat", "sendratari": "seni drama tari", "sept": "september", "seting": "akting", "setting": "atur", "sgt": "sangat", "shg": "sehingga", "siech": "sih", "sikon": "situasi dan kondisi", "simiskin": "miskin", "sinetron": "sinema elektronik", "siramin": "siramkan", "sj": "saja", "skalian": "sekalian", "sklh": "sekolah", "skt": "sakit", "slesai": "selesai", "sll": "selalu", "slma": "selama", "slsai": "selesai", "sma": "sama", "smpt": "sempat", "smw": "semua", "sndiri": "sendiri", "sni": "sini", "sohib": "teman", "soljum": "sholat jumat", "solve": "solusi", "somse": "sombong sekali", "songong": "sombong", "sorry": "maaf", "sory": "maaf", "sosbud": "sosial-budaya", "sosek": "sosial-ekonomi", "sospol": "sosial-politik", "sotoy": "sok tahu", "sowry": "maaf", "spa": "siapa", "spd": "sepeda", "sppa": "siapa", "sprti": "seperti", "spt": "seperti", "spy": "supaya", "srtfkt": "sertifikat", "stelah": "setelah", "stiap": "setiap", "still": "masih", "stlh": "setelah", "subbag": "subbagian", "suk": "masuk", "sumbangin": "sumbangkan", "sumpek": "sempit", "sy": "saya", "syg": "sayang", "syp": "siapa", "tabanas": "tabungan pembangunan nasional", "tajir": "kaya", "tak": "tidak", "tar": "nanti", "tau": "tahu", "tauk": "tahu", "taun": "tahun", "taw": "tahu", "tawh": "tahu", "td": "tadi", "tdi": "tadi", "tdk": "tidak", "te2p": "tetap", "teh": "kakak perempuan", "tekor": "rugi", "telat": "terlambat", "telkom": "telekomunikasi", "telmi": "telat berpikir", "telp": "telepon", "temen": "teman", "temen2": "teman-teman", "tengil": "menyebalkan", "tengok": "menjenguk", "tepar": "terkapar", "terbitin": "terbitkan", "tggu": "tunggu", "tgl": "tanggal", "tgu": "tunggu", "thanks": "terima kasih", "thankz": "terima kasih", "thd": "terhadap", "thn": "tahun", "thx": "terima kasih", "tilang": "bukti pelanggaran", "tilep": "tilap", "tioghoa": "tionghoa", "tipi": "TV", "tipiwan": "TvOne", "tirani": "tiran", "tkg": "tukang", "tks": "terima kasih", "tll": "terlalu", "tlp": "telepon", "tlpn": "telepon", "tls": "tulis", "tman": "teman", "tmbah": "tambah", "tmbh": "tambah", "tmen2": "teman-teman", "tmn2": "teman-teman", "tmpah": "tumpah", "tmph": "tumpah", "tmpt": "tempat", "tnda": "tanda", "tngu": "tunggu", "tnh": "tanah", "tnyta": "ternyata", "togel": "toto gelap", "tokai": "tai", "toserba": "toko serba ada", "tp": "tapi", "tpi": "tapi", "tq": "terima kasih", "trdhulu": "terdahulu", "tren": "trend", "trending": "trend", "trgntg": "tergantung", "trima": "terima kasih", "trims": "terima kasih", "trlalu": "terlalu", "trm": "terima", "trs": "terus", "trutama": "terutama", "tsangka": "tersangka", "tst": "tahu sama tahu", "ttg": "tentang", "tuch": "tuh", "tuir": "tua", "uasai": "kuasa", "ud": "sudah", "udah": "sudah", "udh": "sudah", "ujg": "ujung", "unyu": "lucu", "uplot": "unggah", "urang": "saya", "usah": "perlu", "utk": "untuk", "valas": "valuta asing", "wadir": "wakil direktur", "wamil": "wajib militer", "warkop": "warung kopi", "warteg": "warung tegal", "waspadalah": "waspada", "wat": "buat", "watuk": "batuk", "wkt": "waktu", "wkwkkw": "tertawa", "wtb": "beli", "wtf": "what the fuck", "wts": "jual", "wtt": "tukar", "xixixi": "tertawa", "yap": "iya", "yaudah": "ya sudah", "yawdah": "ya sudah", "yg": "yang", "yowes": "ya sudah", "yup": "iya", "zonk": "bodoh"}''')
len(SLANG)

950

In [6]:
# -> src/preprocess.py
NEGATIONS = {"tidak","tak","bukan","belum","jangan","kurang","tanpa","nggak","ngga","gak","ga",
             "enggak","tdk","gagal","susah","sulit"}
INTENSIFIERS = {"sangat","banget","sekali","terlalu","paling","lebih"}
CONTRAST = {"tapi","tetapi","namun","padahal","cuma","hanya"}
STOPWORDS = {"yang","di","ke","dari","dan","atau","ini","itu","untuk","pada","dengan","dalam","adalah",
             "akan","oleh","saya","aku","kamu","nya","sih","dong","deh","kok","lah","kah","pun","juga",
             "ya","yaa","loh","woi","woy","si","para","sebuah"} - NEGATIONS - INTENSIFIERS - CONTRAST

URL_RE     = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
REPEAT_RE  = re.compile(r"(\D)\1{2,}")                      # bagusssss -> bagus; angka TIDAK dipadatkan (10000 tetap)
EMOJI_CLS  = "\U0001F300-\U0001FAFF\U00002600-\U000027BF"
EMOJI_RE   = re.compile(f"[{EMOJI_CLS}]")
KEEP_RE    = re.compile(f"[^\\w\\s!?{EMOJI_CLS}]")             # buang tanda baca lain, simpan ! ? emoji

def light_clean(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = URL_RE.sub(" ", t); t = MENTION_RE.sub(" ", t)
    t = REPEAT_RE.sub(r"\1", t)
    t = re.sub(r"([!?])\1+", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

def normalize_slang(text):
    return " ".join(SLANG.get(w, w) for w in text.split())

def tokenize(text):
    text = EMOJI_RE.sub(lambda m: f" {m.group(0)} ", text)
    text = re.sub(r"([!?])", r" \1 ", text)
    return KEEP_RE.sub(" ", text).split()

def mark_negation(tokens):
    """[tidak, bagus] -> [tidak, tidak_bagus]. Token negasi asli tetap disimpan."""
    out, i = [], 0
    while i < len(tokens):
        tok = tokens[i]
        if tok in NEGATIONS and i + 1 < len(tokens) and tokens[i+1] not in {"!", "?"} \
                and tokens[i+1] not in STOPWORDS:
            out += [tok, f"{tok}_{tokens[i+1]}"]
            i += 2
        else:
            out.append(tok); i += 1
    return out

def classic_clean(text):
    toks = tokenize(normalize_slang(light_clean(text)))
    toks = [w for w in toks if w not in STOPWORDS]
    return " ".join(mark_negation(toks))

In [7]:
# Uji cepat: kalau salah satu assert gagal, ada yang rusak di preprocessing.
assert "tidak_bagus" in classic_clean("Aplikasi tidak bagus").split()
assert "belum" in classic_clean("belum bisa login").split()
assert "tidak" in classic_clean("gak bisa dibuka").split()
assert "😡" in classic_clean("kecewa 😡") and "!" in classic_clean("parah!!!")
assert light_clean("bagusssss") == "bagus"
assert light_clean("paket 10.000 isi 1000mb") == "paket 10.000 isi 1000mb"   # bug lama: "10.0" dan "10mb"
assert light_clean("parah,,,, 😡😡😡") == "parah, 😡"                     # tanda baca & emoji tetap dipadatkan
for s in ["Bagus banget ya, bayar tapi kuota hilang!!! 😡", "gak bisa login, tidak bagus sama sekali"]:
    print(s, "\n  ->", classic_clean(s))

Bagus banget ya, bayar tapi kuota hilang!!! 😡 
  -> bagus banget bayar tapi kuota hilang ! 😡
gak bisa login, tidak bagus sama sekali 
  -> tidak tidak_bisa login tidak tidak_bagus sama sekali


## 3. Label dari rating + diagnosa
* Rating dipakai sebagai **weak label** (1-2 negatif, 3 netral, 4-5 positif). Lebih tahan sarkasme daripada lexicon, tetapi **tidak seimbang kualitasnya**: pada sampel 500 review dari data ini, bintang 1-2 hampir selalu benar-benar negatif, sedangkan sekitar sepertiga bintang 4-5 isinya keluhan (banyak yang memberi bintang 5 agar ulasan naik ke atas). Karena itu label "positive" dari rating paling berisik. Angka ini dari sampel kecil dan dipengaruhi pembacaan Claude, jadi anggap sebagai perkiraan kasar.
* `conflict_flag` menandai review yang nada teksnya berlawanan dengan rating -> **kandidat sarkasme/ambigu**. Ini penyaring berbasis daftar kata kecil, bukan detektor sarkasme.

In [8]:
# -> src/labeling.py
POS_HINTS = {"bagus","mantap","keren","hebat","terbaik","juara","top","luar biasa","makasih","terima kasih",
             "cepat","lancar","puas","recommended","oke","wow","kece","sukses","bangga"}
NEG_HINTS_V1 = {"jelek","buruk","parah","lemot","lambat","error","gagal","kecewa","sampah","payah","rugi",
                "tidak bisa","gak bisa","ngga bisa","bug","hilang","mahal"}
# Ditambahkan setelah analisis error di gold_val (bagian 6): keluhan ringan/sopan di bintang 4-5
# ("kadang lambat", "cepat habis", "tolong diperbaiki") lolos dari filter lama dan mengajari model
# bahwa nada sopan = positif.
NEG_HINTS = NEG_HINTS_V1 | {"kadang","cepat habis","tolong","perbaiki","diperbaiki","tingkatkan","ditingkatkan",
                            "lemah","tidak stabil","ngelag","lag","susah"}

# Filter C (setelah relabel ulasan campuran, 2026-10-27): aturan gold sekarang "ada keluhan konkret -> negative",
# jadi review bintang 4-5 berpola "pujian TAPI keluhan" juga dianggap positif berisik dan tidak dipakai melatih.
CONTRAST_HINTS = {"tapi","tp","tpi","tetapi","namun","walaupun","walau","meskipun","cuma","sayang","sayangnya","padahal"}
FILTER_VERSION = "C: NEG_HINTS diperluas + campuran bintang 4-5 (kontras + keluhan) dibuang"

def rating_to_label(score):
    if score == 3:
        return None if TWO_CLASS else "neutral"   # bintang 3 = ambigu, bukan label
    return "positive" if score >= 4 else "negative"

def _has_any(text, hints):
    return any(re.search(rf"\b{re.escape(h)}\b", text) for h in hints)

def conflict_flag(text, score, neg_hints=None, contrast_rule=True):
    pos, neg = _has_any(text, POS_HINTS), _has_any(text, NEG_HINTS if neg_hints is None else neg_hints)
    if contrast_rule and score >= 4 and neg and _has_any(text, CONTRAST_HINTS):
        return True                    # "sinyal bagus tapi aplikasi lag parah" bintang 5
    if score <= 2 and pos:
        return True                    # "bagus banget ya, sinyal hilang terus" bintang 1
    if score >= 4 and neg and not pos:
        return True                    # bintang 5 tapi isinya keluhan
    return False

# -> src/make_dataset.py
def build_dataset(raw):
    df = raw.dropna(subset=["content", "score"]).copy()
    df["content"] = df["content"].astype(str)
    key = df["content"].str.lower().str.strip()
    n0 = len(df); df = df[~key.duplicated()]
    print(f"Dedup teks: {n0} -> {len(df)}")
    df["text_light"]   = df["content"].map(light_clean)
    df["text_classic"] = df["content"].map(classic_clean)
    df = df[df["text_light"].str.len() > 0].copy()
    df["score"] = df["score"].astype(int)
    df["label_rating"]  = df["score"].map(rating_to_label)
    if TWO_CLASS:
        n3 = int(df["label_rating"].isna().sum())
        df = df.dropna(subset=["label_rating"]).copy()
        print(f"Bintang 3 dikeluarkan (label rating ambigu): {n3} baris")
    df["conflict_flag"] = [conflict_flag(t, s) for t, s in zip(df["text_light"], df["score"])]
    return df.reset_index(drop=True)

In [9]:
df = build_dataset(raw)
print(df["label_rating"].value_counts(normalize=True).round(3))
print("conflict_flag:", int(df["conflict_flag"].sum()), "baris")
pd.crosstab(df["score"], df["conflict_flag"], margins=True)

Dedup teks: 12000 -> 10175


Bintang 3 dikeluarkan (label rating ambigu): 558 baris


label_rating
negative    0.794
positive    0.206
Name: proportion, dtype: float64
conflict_flag: 1127 baris


conflict_flag,False,True,All
score,,,
1,6229,553,6782
2,754,97,851
4,213,124,337
5,1294,353,1647
All,8490,1127,9617


In [10]:
pd.set_option("display.max_colwidth", 200)
df[df["conflict_flag"]].sample(min(15, int(df["conflict_flag"].sum())), random_state=1)[["content", "score"]]

,content,score
3150,"sebaiknya klo ada yg mau pake Telkomssel pikir2 lagi dah... udah mah mahal paket internetnya,jaringannya makin buruk,harga paket internet dalem setahun bisa berkali2 naiknya,tapi ga sesuai sama ha...",1
9436,knp aplikasinya kalau dibuka malah langsung keluar sementara signal bagus,1
4070,"APLIKASI RUSAK UDAH DI UPDATE MASIH BERMASALAH JARINGAN BAGUS NI APLIKASI YG RUSAK APA DEVELOVER NYA YG TOLOL APLIKASI SAMPAH GINI BELI PAKET SELALU BERMASALAH MMNG ANEH SI , GK BISA BIKIN APK MEN...",1
9585,"setiap bukak aplikasi ngelag terus, padahal jaringan bagus,udah update segala macam tetap aja ngeleg,tolong di perbaiki apk nya",1
4736,"mohon perbaiki kualitas jaringan nya , percuma apk keren tapi jaringan selalu banyak Drama",2
409,"Sore Kak kenapa Akhir akhir bulan ini Pulsa saya terus Terpotong,padahal saya tidak menggunakan paket internet Telkomsel.Tolong dong Diperbaiki,Paket Internetnya sekarang Mahal nanti pelanggan pin...",1
722,"udah dicoba semua yang disaranin masih tetep aja jaringan nya ilang ilangan dari dulu kaga ada perubahan sama sekali, padahal ini kartu nya beli yang baru masih sama aja yang ada update terus apli...",5
9528,"apk nya bagus,tapi ada kendala agak lemot, tolong diperbaiki lagi ya min.",5
2077,haduh bukan nya makin bagus malah makin payah aplikasi nya masa lama kali reload apk nya,1
7588,"burukkm sangattt burukk. lemotttt aplikasi nya. mau beli kuota pun sulit, jaringan bagus tapi loading aplikasi lemot parah gabisa masuk payment. DARI DULU GA PERNAH ADA PERBAIKAN. ANEH",1


## 4. Gold set berbantuan (label yang dipakai untuk menilai model)
Angka akurasi hanya bermakna jika dibandingkan dengan label yang benar. Pelabelan 500 baris sepenuhnya manual memberatkan, jadi dipakai cara **berbantuan dengan kalibrasi**. Pelabelan sudah selesai (Oktober 2026); tabel berikut menjelaskan peran setiap file.

| File (folder `data/`) | Isi | Peran dalam gold set |
|---|---|---|
| `1_gold_blind_100.csv` | 100 ulasan yang dilabeli manusia **secara independen** (tanpa melihat rating maupun label LLM); 98 terisi | Patokan untuk mengukur seberapa bisa dipercaya label LLM (kalibrasi) |
| `2_gold_review_400.csv` | 400 ulasan dengan pra-label LLM dan kolom `verdict` dari manusia | Manusia memeriksa baris `PERIKSA` (label LLM ragu) dan `AUDIT` (sampel acak dari label LLM yang yakin); baris `OPSIONAL` tidak diperiksa |
| `3_llm_prelabel_500.csv` | Pra-label LLM (Claude) untuk ke-500 ulasan, beserta teks, rating, alasan, dan tingkat keyakinan | Dibandingkan dengan file 1 untuk kalibrasi; sumber teks dan rating gold. Selama pelabelan independen di file 1, file ini sengaja tidak dibuka |
| `4_relabel_campuran.csv` | 116 ulasan campuran yang dilabeli ulang manusia | Menimpa label lama dengan aturan di `4_PANDUAN_relabel_campuran.md` |

**Format kolom** (jika ingin menambah atau memperbaiki label): file 1, `label` = `negative` / `neutral` / `positive` (boleh `negatif/netral/positif`) dan `sarcasm` = `1` jika sarkastik, `0` jika tidak. File 2, kolom `verdict`: `ok` = setuju dengan label LLM; atau label yang benar (`negative/neutral/positive`); `skip` = tidak bisa dinilai; kosong = tidak diperiksa. Jika sarkasmenya berbeda, isi `final_sarcasm` (0/1). File 4, kolom `label_baru`.

**Mengapa begini:** label independen manusia di file 1 dipakai untuk mengukur seberapa sering label LLM sama dengan label manusia (bagian kalibrasi di bawah). Baris AUDIT adalah sampel acak dari baris "yakin"; jumlah koreksi di sana memperkirakan seberapa sering baris OPSIONAL yang tidak diperiksa salah. Hasil akhirnya (`data/gold_final.csv`) memberi tiap baris kolom `source` (`human_blind`, `human_checked`, `human_relabel`, `claude_unchecked`), dan evaluasi dilaporkan juga khusus baris yang sudah dilihat manusia.

**Panduan pelabelan (supaya konsisten):**
* `negative` = keluhan / kekecewaan terhadap Telkomsel atau aplikasinya (termasuk sinyal dan harga). `positive` = puas / memuji. `neutral` = faktual, campuran seimbang, saran tanpa nada emosi, atau tidak jelas. **Mode 2 kelas (`TWO_CLASS = True`, bawaan): `neutral` otomatis dihitung `positive`, jadi `positive` berarti BUKAN keluhan.**
* Nilai **nada teks**, bukan rating. Review sarkastik ("bagus banget ya, kuota hilang terus") = `negative` dengan `sarcasm = 1`.
* Review campuran ("jaringan bagus tapi aplikasi sering error"): **ada masalah nyata yang belum selesai -> `negative`**, walaupun ada pujian (aturan sejak relabel 27 Okt 2026; sebelumnya "pilih sentimen yang dominan"). Detail dan contoh: `data/4_PANDUAN_relabel_campuran.md`.
* Ragu? Tulis alasan singkat di `notes` dan tetap pilih satu label.

**Jalur manual penuh (tanpa bantuan Claude), mis. setelah scraping baru:** set `MAKE_MANUAL_SHEET = True` di sel di bawah, isi `data/gold_manual_sheet.csv`, lalu simpan sebagai `data/gold_final.csv` (kolom `reviewId, content, label, sarcasm`).

In [11]:
# -> src/labeling.py
LABEL_ALIASES = {"positif": "positive", "pos": "positive", "negatif": "negative", "neg": "negative",
                 "netral": "neutral", "neu": "neutral", "net": "neutral"}
if TWO_CLASS:   # netral tanpa keluhan -> positive (bukan-negatif)
    LABEL_ALIASES.update({"neutral": "positive", "netral": "positive", "neu": "positive", "net": "positive"})
OK_WORDS = {"ok", "ya", "y", "x", "v", "setuju", "benar", "oke"}
SKIP_WORDS = {"skip", "buang", "hapus", "tidak_jelas"}   # teks tak bisa dinilai -> dikeluarkan dari gold

def norm_labels(series):
    return series.fillna("").astype(str).str.strip().str.lower().replace(LABEL_ALIASES)

def to01(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(int).clip(0, 1)

def make_gold_sheet(df, n=500, flagged_share=0.4, seed=SEED):
    """Jalur manual penuh. Sampel: sebagian dari kandidat konflik + sisanya stratified per rating."""
    n_flag = min(int(n * flagged_share), int(df["conflict_flag"].sum()))
    flagged = df[df["conflict_flag"]].sample(n_flag, random_state=seed) if n_flag else df.iloc[:0]
    rest = df.drop(flagged.index)
    per = max(1, (n - len(flagged)) // 5)
    parts = [g.sample(min(per, len(g)), random_state=seed) for _, g in rest.groupby("score")]
    gold = pd.concat([flagged, *parts]).sample(frac=1, random_state=seed).head(n)
    sheet = gold[["reviewId", "content"]].copy()
    sheet["label"], sheet["sarcasm"], sheet["notes"] = "", "", ""
    return sheet.reset_index(drop=True)

def load_gold(path=GOLD_CSV):
    """Baca gold final. Kolom wajib: reviewId, content, label, sarcasm. Opsional: score, source."""
    g = pd.read_csv(path, dtype={"reviewId": str})
    lab = norm_labels(g["label"])
    bad = g[(lab != "") & ~lab.isin(LABELS)]
    if len(bad):
        print(f"[PERINGATAN] {len(bad)} baris berlabel tidak dikenali dan DIABAIKAN, contoh: "
              f"{bad['label'].astype(str).unique()[:5].tolist()}. Gunakan: negative / neutral / positive.")
    g = g.assign(label=lab)[lab.isin(LABELS)].copy()
    g["sarcasm"] = to01(g["sarcasm"])
    if "source" not in g.columns:
        g["source"] = "human_manual"
    return g.reset_index(drop=True)

def apply_relabel(gold, path=GOLD_RELABEL):
    """Timpa label gold dengan hasil pelabelan ulang ulasan campuran (file 4), jika ada."""
    if not Path(path).exists():
        return gold
    rl = pd.read_csv(path, dtype=str, encoding="utf-8-sig")
    v = rl["label_baru"].fillna("").astype(str).str.strip().str.lower()
    skip_ids = set(rl.loc[v.isin(SKIP_WORDS), "reviewId"])
    new = norm_labels(rl["label_baru"])
    rl = rl.assign(label_baru=new)[new.isin(LABELS)]
    unknown = int(((v != "") & ~v.isin(SKIP_WORDS) & ~new.isin(LABELS)).sum())
    if unknown:
        print(f"[PERINGATAN] {unknown} label_baru tidak dikenali di {Path(path).name}; diabaikan.")
    gold = gold.copy()
    m = gold["reviewId"].map(rl.set_index("reviewId")["label_baru"])
    changed = m.notna() & (m != gold["label"])
    print(f"\n=== RELABEL ulasan campuran ({Path(path).name}) ===")
    print(f"baris dilabeli ulang: {int(m.notna().sum())} | label berubah: {int(changed.sum())} | skip: {len(skip_ids)}")
    if changed.any():
        print(pd.crosstab(gold.loc[changed, "label"], m[changed], rownames=["label_lama"], colnames=["label_baru"]))
    gold.loc[m.notna(), "label"] = m[m.notna()]
    gold.loc[m.notna(), "source"] = "human_relabel"
    return gold[~gold["reviewId"].isin(skip_ids)].reset_index(drop=True)

def assemble_gold(blind_path=GOLD_BLIND, review_path=GOLD_REVIEW, prelabel_path=CLAUDE_PRELABEL,
                  out_path=GOLD_CSV, min_blind=50):
    """Kalibrasi label LLM (Claude) terhadap label independen manusia, lalu susun data/gold_final.csv."""
    from sklearn.metrics import cohen_kappa_score
    blind = pd.read_csv(blind_path, dtype=str)
    review = pd.read_csv(review_path, dtype=str)
    pre = pd.read_csv(prelabel_path, dtype=str)
    pre["claude_label"] = norm_labels(pre["claude_label"])
    review["claude_label"] = norm_labels(review["claude_label"])
    blind["label"] = norm_labels(blind["label"])
    done = blind[blind["label"].isin(LABELS)].copy()
    done["sarcasm"] = to01(done["sarcasm"])
    print(f"File 1 (buta): {len(done)}/{len(blind)} baris terisi.")
    if len(done) < min_blind:
        print(f"-> Isi minimal {min_blind} baris di {Path(blind_path).name} dulu; gold belum disusun.")
        return None

    # ---------- kalibrasi: label Anda vs label Claude ----------
    cal = done.merge(pre[["reviewId", "claude_label", "claude_sarcasm", "claude_reason", "blind_kind"]], on="reviewId")
    cal["agree"] = cal["label"] == cal["claude_label"]
    print("\n=== KALIBRASI: seberapa sering Claude sama dengan Anda ===")
    rnd = cal[cal["blind_kind"] == "random"]
    if len(rnd):
        k = cohen_kappa_score(rnd["label"], rnd["claude_label"]) if rnd["label"].nunique() > 1 else float("nan")
        print(f"Sampel ACAK (n={len(rnd)}, perkiraan paling jujur): sepakat {rnd['agree'].mean():.0%} | kappa={k:.2f}")
    print(f"Semua baris buta (n={len(cal)}): sepakat {cal['agree'].mean():.0%}")
    print("Jika Claude bilang X, berapa persen Anda juga bilang X:")
    print(cal.groupby("claude_label")["agree"].agg(["mean", "count"]).rename(columns={"mean": "sepakat", "count": "n"}).round(2))
    cs = cal["claude_sarcasm"].astype(int)
    print(f"Sarkasme: Anda={int(cal['sarcasm'].sum())}, Claude={int(cs.sum())}, keduanya={int(((cal['sarcasm']==1)&(cs==1)).sum())}")
    dis = cal[~cal["agree"]]
    if len(dis):
        print(f"\nBaris yang berbeda ({len(dis)}): baca dan putuskan siapa yang benar")
        display(dis[["content", "label", "claude_label", "claude_reason"]].head(25))

    # ---------- file 2: verdict ----------
    v = review["verdict"].fillna("").astype(str).str.strip().str.lower()
    n_skip = int(v.isin(SKIP_WORDS).sum())
    if n_skip:
        print(f"{n_skip} baris ditandai skip (tidak bisa dinilai) -> dikeluarkan dari gold")
    review, v = review[~v.isin(SKIP_WORDS)].copy(), v[~v.isin(SKIP_WORDS)]
    ov = v.replace(LABEL_ALIASES)
    is_ok, is_over = v.isin(OK_WORDS), ov.isin(LABELS)
    unknown = review[(v != "") & ~is_ok & ~is_over]
    if len(unknown):
        print(f"\n[PERINGATAN] {len(unknown)} verdict tidak dikenali dan dianggap KOSONG: {unknown['verdict'].unique()[:5].tolist()}")
    review["label"] = np.where(is_over, ov, review["claude_label"])
    review["source"] = np.where(is_ok | is_over, "human_checked", "claude_unchecked")
    fs = review["final_sarcasm"].fillna("").astype(str).str.strip()
    review["sarcasm"] = np.where(fs != "", to01(fs), review["claude_sarcasm"].astype(int))
    print("\n=== PEMERIKSAAN file 2 ===")
    rep = review.assign(checked=is_ok | is_over, overridden=is_over).groupby("tugas").agg(
        baris=("reviewId", "count"), diperiksa=("checked", "sum"), dikoreksi=("overridden", "sum"))
    rep["koreksi_per_diperiksa"] = (rep["dikoreksi"] / rep["diperiksa"].replace(0, np.nan)).round(2)
    print(rep)
    au = rep.loc["AUDIT"] if "AUDIT" in rep.index else None
    if au is not None and au["diperiksa"] >= 20:
        print(f"-> Di baris AUDIT (yakin tinggi) Anda mengoreksi {au['koreksi_per_diperiksa']:.0%}. "
              f"Itu perkiraan kasar tingkat salah pada baris OPSIONAL yang tidak Anda lihat.")
    elif au is not None:
        print("-> Periksa lebih banyak baris AUDIT agar perkiraan tingkat salah OPSIONAL berarti (>= 20).")

    # ---------- susun gold final ----------
    part1 = done.assign(source="human_blind")[["reviewId", "label", "sarcasm", "source"]]
    part2 = review[["reviewId", "label", "sarcasm", "source"]]
    gold = pd.concat([part1, part2]).merge(pre[["reviewId", "content", "score"]], on="reviewId", how="left")
    gold["score"] = pd.to_numeric(gold["score"], errors="coerce")
    gold = apply_relabel(gold)
    gold.to_csv(out_path, index=False)
    print(f"\nGold final: {len(gold)} baris -> {out_path}")
    print(gold["source"].value_counts().to_string())
    return gold

def split_gold(gold, path=GOLD_SPLIT_CSV, test_size=0.5, seed=SEED):
    """Bagi gold menjadi val (untuk memilih/menganalisis) dan test (dilaporkan SEKALI di akhir).

    Pembagian disimpan di `path` dan dipakai ulang. Baris gold baru (yang belum punya split)
    dibagi stratified lalu DITAMBAHKAN, sehingga baris lama tidak pernah pindah antara val dan test.
    """
    gold = gold.copy()
    gold["reviewId"] = gold["reviewId"].astype(str)
    old = pd.read_csv(path, dtype=str) if Path(path).exists() else pd.DataFrame(columns=["reviewId", "split"])
    new = gold[~gold["reviewId"].isin(old["reviewId"])]
    if len(new):
        key = new["label"] + "|" + new["source"]
        key = key.where(key.map(key.value_counts()) >= 2, new["label"])   # grup kecil -> stratify per label saja
        val_ids, test_ids = train_test_split(new["reviewId"], test_size=test_size, random_state=seed, stratify=key)
        add = pd.DataFrame({"reviewId": list(val_ids) + list(test_ids),
                            "split": ["val"] * len(val_ids) + ["test"] * len(test_ids)})
        old = pd.concat([old, add], ignore_index=True)
        old.to_csv(path, index=False)
        print(f"Split gold: {len(new)} baris baru dibagi -> {path}")
    gold = gold.merge(old, on="reviewId", how="left")
    return gold[gold["split"] == "val"].reset_index(drop=True), gold[gold["split"] == "test"].reset_index(drop=True)


In [12]:
MAKE_MANUAL_SHEET = False   # True = buat lembar untuk pelabelan manual penuh (mis. setelah scraping baru)
if MAKE_MANUAL_SHEET:
    make_gold_sheet(df, n=500).to_csv(DATA_DIR / "gold_manual_sheet.csv", index=False)

if GOLD_BLIND.exists() and GOLD_REVIEW.exists() and CLAUDE_PRELABEL.exists():
    assemble_gold()
else:
    print("File gold berbantuan tidak lengkap di folder data/ -> memakai gold_final.csv jika sudah ada.")

gold = load_gold(GOLD_CSV) if GOLD_CSV.exists() else None
gold_val = gold_test = None
if gold is None or len(gold) == 0:
    print("Gold belum ada -> model tetap dilatih, tetapi TIDAK ada angka yang bisa dipercaya.")
    gold = None
else:
    print(f"\nGold siap: {len(gold)} baris | sarkasme: {int(gold['sarcasm'].sum())}")
    print(gold["label"].value_counts().to_string())
    print(gold["source"].value_counts().to_string())
    gold_val, gold_test = split_gold(gold)
    summary = pd.concat({"val": gold_val.groupby(["label", "source"]).size(),
                         "test": gold_test.groupby(["label", "source"]).size()}, axis=1).fillna(0).astype(int)
    print(f"\ngold_val={len(gold_val)}  gold_test={len(gold_test)}  "
          f"(sarkasme val={int(gold_val['sarcasm'].sum())}, test={int(gold_test['sarcasm'].sum())})")
    print(summary.to_string())

File 1 (buta): 98/100 baris terisi.

=== KALIBRASI: seberapa sering Claude sama dengan Anda ===
Sampel ACAK (n=50, perkiraan paling jujur): sepakat 98% | kappa=0.90
Semua baris buta (n=98): sepakat 94%
Jika Claude bilang X, berapa persen Anda juga bilang X:
              sepakat   n
claude_label             
negative         0.98  64
positive         0.85  34
Sarkasme: Anda=20, Claude=14, keduanya=10

Baris yang berbeda (6): baca dan putuskan siapa yang benar


,content,label,claude_label,claude_reason
12,"berikan kouta lebih murah lagii yaa dan jangan mahal mahal harganyaa , terimakasih",positive,negative,NaN
51,"baik, cuma terlalu boros makan paket",negative,positive,"""baik, cuma boros paket"" campuran ringan"
57,Sudah bisa di buka aplikasi nya. Walaupun sangat lemod.,negative,positive,ada yang berfungsi tapi lemot; campuran
67,"mantaap,,, tp sungguh sayang kalau mati PLN jaringan nya hilang",negative,positive,memuji paket lalu 'sayang jaringan hilang kalau mati PLN'; campuran
85,tolong d tingkat kan lagi,negative,positive,permintaan singkat tanpa nada jelas
88,"bisa liat paket,tapi kadang kadang agak error",negative,positive,'bisa liat paket' tapi 'agak error'; campuran ringan


2 baris ditandai skip (tidak bisa dinilai) -> dikeluarkan dari gold

=== PEMERIKSAAN file 2 ===
          baris  diperiksa  dikoreksi  koreksi_per_diperiksa
tugas                                                       
AUDIT        50         50          0                   0.00
OPSIONAL    290          0          0                    NaN
PERIKSA      58         58          4                   0.07
-> Di baris AUDIT (yakin tinggi) Anda mengoreksi 0%. Itu perkiraan kasar tingkat salah pada baris OPSIONAL yang tidak Anda lihat.

=== RELABEL ulasan campuran (4_relabel_campuran.csv) ===
baris dilabeli ulang: 116 | label berubah: 4 | skip: 0
label_baru  negative
label_lama          
positive           4

Gold final: 496 baris -> data/gold_final.csv
source
claude_unchecked    216
human_relabel       116
human_checked        88
human_blind          76

Gold siap: 496 baris | sarkasme: 22
label
negative    427
positive     69
source
claude_unchecked    216
human_relabel       116
human_checked 

## 5. Baseline: TF-IDF (kata + karakter) + Logistic Regression
Seluruh gold (val + test) dikeluarkan dari data training agar tidak bocor.
**Angka utama = blok GOLD_VAL.** `gold_test` tidak disentuh di sini; dilaporkan sekali di bagian 9.
Blok VAL (label rating) hanya menunjukkan kesepakatan model dengan label rating - jangan dikutip sebagai akurasi.

In [13]:
# -> src/evaluate.py
def evaluate(y_true, y_pred, name, slices=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    res = {"name": name, "n": int(len(y_true)),
           "accuracy": float(accuracy_score(y_true, y_pred)),
           "macro_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
           "per_class": classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0),
           "confusion_matrix": confusion_matrix(y_true, y_pred, labels=LABELS).tolist(), "slices": {}}
    for sname, mask in (slices or {}).items():
        mask = np.asarray(mask, dtype=bool)
        if mask.sum() == 0:
            continue
        present = [l for l in LABELS if l in set(y_true[mask])]
        res["slices"][sname] = {"n": int(mask.sum()),
            "accuracy": float(accuracy_score(y_true[mask], y_pred[mask])),
            "macro_f1": float(f1_score(y_true[mask], y_pred[mask], labels=present, average="macro", zero_division=0))}
    return res

THRESHOLD_GRID = np.round(np.arange(0.30, 0.91, 0.05), 2)

def choose_threshold(y_rating, proba_pos, grid=THRESHOLD_GRID):
    """Threshold positif dipilih di VAL label rating (BUKAN di gold), lalu dipakai apa adanya di gold."""
    y_rating = np.asarray(y_rating)
    f1m = lambda t: f1_score(y_rating, np.where(proba_pos >= t, "positive", "negative"),
                             labels=LABELS, average="macro", zero_division=0)
    return float(max(grid, key=f1m))

def to_label(proba_pos, threshold):
    return np.where(np.asarray(proba_pos) >= threshold, "positive", "negative")

def print_report(res):
    print(f"\n=== {res['name']}  (n={res['n']}) ===")
    print(f"accuracy : {res['accuracy']:.3f}\nmacro-F1 : {res['macro_f1']:.3f}   <- angka utama")
    for lab in LABELS:
        m = res["per_class"].get(lab, {})
        print(f"  {lab:9s} P={m.get('precision',0):.2f} R={m.get('recall',0):.2f} "
              f"F1={m.get('f1-score',0):.2f} support={int(m.get('support',0))}")
    print(pd.DataFrame(res["confusion_matrix"], index=[f"benar:{l}" for l in LABELS],
                       columns=[f"pred:{l}" for l in LABELS]))
    for s, v in res["slices"].items():
        print(f"  irisan[{s}] n={v['n']} acc={v['accuracy']:.3f} macroF1={v['macro_f1']:.3f}")

# -> src/train_baseline.py
def build_pipeline(C=2.0):
    feats = ColumnTransformer([
        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"[^\s]+"), "text_classic"),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True,
                                 max_features=150_000), "text_light"),
    ])
    return Pipeline([("features", feats),
                     ("clf", LogisticRegression(C=C, max_iter=3000, class_weight="balanced", random_state=SEED))])

def norm_key(series):
    return series.astype(str).str.lower().str.strip()

def split_data(df, gold, drop_noisy_positives=DROP_NOISY_POSITIVES):
    # Keluarkan gold dari TRAINING (berdasarkan reviewId DAN isi teks, agar duplikat dari scraping baru tidak bocor).
    if gold is not None:
        df = df[~df["reviewId"].astype(str).isin(gold["reviewId"].astype(str)) & ~norm_key(df["content"]).isin(set(norm_key(gold["content"])))]
    if drop_noisy_positives:   # bintang 4-5 tapi teks penuh kata keluhan -> label rating hampir pasti keliru
        noisy = (df["score"] >= 4) & df["conflict_flag"]
        print(f"Dibuang dari data latih (positif berisik): {int(noisy.sum())} baris")
        df = df[~noisy]
    return train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["label_rating"])

In [14]:
train, val = split_data(df, gold)
print(f"train={len(train)}  val={len(val)}  gold={0 if gold is None else len(gold)} (dikeluarkan dari training)")

pipe = build_pipeline().fit(train, train["label_rating"])
POS_IDX = list(pipe.classes_).index("positive")
THRESHOLD = choose_threshold(val["label_rating"], pipe.predict_proba(val)[:, POS_IDX])
print(f"Threshold baseline (dipilih di VAL rating): {THRESHOLD}")
results = [{"name": "config", "threshold": THRESHOLD, "filter_version": FILTER_VERSION}]

r = evaluate(val["label_rating"], to_label(pipe.predict_proba(val)[:, POS_IDX], THRESHOLD), "VAL (label rating) - hanya referensi",
             {"conflict_flag": val["conflict_flag"].to_numpy()})
print_report(r); results.append(r)

def prepare_gold_eval(gold, df):
    # Hitung fitur dari teks gold sendiri (tidak bergantung pada data scraping terbaru).
    g = gold.copy()
    g["text_light"] = g["content"].map(light_clean)
    g["text_classic"] = g["content"].map(classic_clean)
    if "score" not in g.columns or g["score"].isna().all():
        g["score"] = g["reviewId"].map(df.set_index("reviewId")["score"])
    g["conflict_flag"] = [bool(conflict_flag(t, s)) if pd.notna(s) else False for t, s in zip(g["text_light"], g["score"])]
    return g

def eval_gold_split(gdf, pred_col, split_name, model_name):
    """Laporan untuk satu split gold: semua baris + hanya baris yang dilihat manusia."""
    out = []
    r = evaluate(gdf["label"], gdf[pred_col], f"{split_name} {model_name} (semua baris)",
                 {"sarkasme": gdf["sarcasm"].to_numpy() == 1, "conflict_flag": gdf["conflict_flag"].to_numpy()})
    print_report(r); out.append(r)
    human = gdf[gdf["source"] != "claude_unchecked"]
    if len(human) and len(human) < len(gdf):
        r2 = evaluate(human["label"], human[pred_col], f"{split_name} {model_name} (hanya dilihat manusia)",
                      {"sarkasme": human["sarcasm"].to_numpy() == 1})
        print_report(r2); out.append(r2)
    return out

gold_val_df = gold_test_df = None
if gold is not None:
    gold_val_df = prepare_gold_eval(gold_val, df)
    gold_val_df["proba_pos"] = pipe.predict_proba(gold_val_df)[:, POS_IDX]
    gold_val_df["pred"] = to_label(gold_val_df["proba_pos"], THRESHOLD)
    results += eval_gold_split(gold_val_df, "pred", "GOLD_VAL", "baseline")

joblib.dump(pipe, MODEL_DIR / "baseline_tfidf_lr.joblib")
json.dump(results, open(MODEL_DIR / "baseline_report.json", "w"), indent=2, ensure_ascii=False)

Dibuang dari data latih (positif berisik): 386 baris
train=7474  val=1320  gold=496 (dikeluarkan dari training)


Threshold baseline (dipilih di VAL rating): 0.6

=== VAL (label rating) - hanya referensi  (n=1320) ===
accuracy : 0.942
macro-F1 : 0.892   <- angka utama
  negative  P=0.96 R=0.97 F1=0.97 support=1107
  positive  P=0.84 R=0.80 F1=0.82 support=213
                pred:negative  pred:positive
benar:negative           1074             33
benar:positive             43            170
  irisan[conflict_flag] n=72 acc=0.958 macroF1=0.979

=== GOLD_VAL baseline (semua baris)  (n=248) ===
accuracy : 0.948
macro-F1 : 0.893   <- angka utama
  negative  P=0.98 R=0.96 F1=0.97 support=215
  positive  P=0.76 R=0.88 F1=0.82 support=33
                pred:negative  pred:positive
benar:negative            206              9
benar:positive              4             29
  irisan[sarkasme] n=12 acc=0.833 macroF1=0.455
  irisan[conflict_flag] n=120 acc=0.975 macroF1=0.694

=== GOLD_VAL baseline (hanya dilihat manusia)  (n=145) ===
accuracy : 0.931
macro-F1 : 0.866   <- angka utama
  negative  P=0.97 R=0.9

## 6. Analisis error (hanya gold_val)
Analisis dilakukan **hanya pada gold_val**, supaya gold_test tetap menjadi penilaian yang tidak dipengaruhi keputusan kita.
Daftar error disimpan ke `data/errors_gold_val.csv`.

Catatan jujur: sebelum gold dipisah (run 21 Okt), sel ini sempat menampilkan sebagian error dari seluruh 496 baris,
termasuk baris yang sekarang masuk gold_test. Tidak ada keputusan model yang dibuat dari tampilan itu.

In [15]:
if gold_val_df is not None:
    err = gold_val_df[gold_val_df["label"] != gold_val_df["pred"]].copy()
    print(f"Salah: {len(err)} dari {len(gold_val_df)} ({len(err)/len(gold_val_df):.1%})")
    print(pd.crosstab(err["label"], err["source"], margins=True).to_string())
    display_cols = ["content", "label", "pred", "proba_pos", "sarcasm", "score", "source"]
    err[["reviewId"] + display_cols].sort_values(["label", "proba_pos"]).to_csv(DATA_DIR / "errors_gold_val.csv", index=False)
    with pd.option_context("display.max_colwidth", 160):
        print("\n-- Salah pada review sarkastik --")
        display(err[err["sarcasm"] == 1][display_cols])
        print("-- Salah pada review non-sarkastik --")
        display(err[err["sarcasm"] == 0][display_cols])
else:
    print("Isi gold set dulu (bagian 4).")

Salah: 13 dari 248 (5.2%)
source    claude_unchecked  human_blind  human_checked  human_relabel  All
label                                                                     
negative                 3            1              3              2    9
positive                 0            2              2              0    4
All                      3            3              5              2   13

-- Salah pada review sarkastik --


,content,label,pred,proba_pos,sarcasm,score,source
16,"jaringannya sangat bagus,saya sering mengeluarkan kata"" mutiara gara"" sinyal nya E,sehingga saya sering istigfar,ini jaringan sangat bagus beri A+",negative,positive,0.765946,1,1,human_blind
33,pas,positive,negative,0.252395,1,2,human_blind


-- Salah pada review non-sarkastik --


,content,label,pred,proba_pos,sarcasm,score,source
12,"terimakasih sudah memperbaikinya, sekarang tidak perlu waktu lama untuk mengaksesnya, 👍👍👍👍 tapi nanti kalau mulai lemot lagi aku rubah ulasannya,🫡",positive,negative,0.513383,0,4,human_blind
50,low respon.,negative,positive,0.758822,0,1,human_checked
51,aku cinta Telkomsel sampai mati,positive,negative,0.419078,0,4,human_checked
54,"paket cepat habis, signal kadang lemah. thanks 🙏😇🙏",negative,positive,0.851658,0,4,human_checked
65,Kami mengusulkan agar Telkomsel menghadirkan kembali paket dengan spesifikasi sebagai berikut: Harga: Rp50.000 / 30 hari Fitur utama: Unlimited aPromo di My...,positive,negative,0.452393,0,2,human_checked
95,saya top up 105.000 tapi kok cuma 85.000 kemana pulsa saya dasar MARUK,negative,positive,0.715244,0,1,human_checked
120,setelah dari 30mbps dapat promo upgrade ke 200mbps kok tetap kluwer-kluwer saja....,negative,positive,0.751771,0,2,claude_unchecked
148,sangat mudah dalam pembelian Kouta lancar dan tidak lemot sangat mudah di gunakan,negative,positive,0.762814,0,5,human_relabel
178,"Sekarang untuk membuka Aplikasi My Telkomsel butuh kesabaran yang extra, mohon pihak provider Aplikasinya agar mudah digunakan biar pelanggan jadi lebih nya...",negative,positive,0.777594,0,1,claude_unchecked
212,sekarg telkomsel palung cepat habis paket data,negative,positive,0.860279,0,4,claude_unchecked


### Kesimpulan analisis error (baseline v1, gold_val, 2026-10-27)

Baseline v1 (filter lama) salah pada **25 dari 248** ulasan gold_val. Arah kesalahannya berat sebelah:
**20 keluhan diprediksi positif**, hanya 5 sebaliknya. Kategori di bawah adalah hasil pembacaan Claude atas tiap baris,
dan tersimpan di `data/errors_gold_val_v1_baseline.csv` (kolom `kategori_claude`).

| Pola | n | Contoh | Penyebab yang diduga |
|---|---:|---|---|
| Keluhan ringan / sopan | 11 | "aplikasinya kadang lambat", "gagal untuk tukarkan koin", "paket cepat habis, signal kadang lemah. thanks 🙏" | Ulasan bintang 4-5 seperti ini lolos filter positif berisik, sehingga nada sopan ("tolong", "🙏", "thanks") dan kata "cepat" dipelajari sebagai positif |
| Campuran pujian + keluhan | 7 | "jaringannya walaupun bagus tapi ngleg", "makin bagus tampilan… tapi poin hangus" | Batas "sentimen dominan" memang tipis; TF-IDF tidak menangkap bagian mana yang lebih berat |
| Sarkasme / ironi | 3 | "sengaja kasih bintang 5 biar diatas…", "jaringannya sangat bagus… sering istigfar" | Kata pujian literal mendominasi fitur |
| Negasi / idiom | 3 | "kurang bagus", "harga tetap terjangkau, gak naik" (usulan), "cinta Telkomsel sampai mati" | Fitur kata tidak membaca cakupan negasi / idiom |
| Lainnya | 1 | "top up 105.000 tapi kok cuma 85.000… MARUK" | Keluhan berbasis angka, kata "maruk" jarang |

**Tindakan:** filter positif berisik diperluas dengan kata keluhan ringan (`NEG_HINTS`, sel bagian 3). Hasilnya ada di
tabel sel berikut. Threshold tidak diubah dari 0,5, karena threshold yang dipilih di VAL rating tidak memberi tambahan
setelah filter diperbaiki. Kolom `VAL_rating_F1` tidak sebanding antar varian (filter B juga membuang lebih banyak
baris dari VAL); yang sebanding hanya kolom GOLD_VAL.

**Catatan kejujuran:** kata-kata filter dipilih setelah membaca error gold_val, jadi angka gold_val setelah perbaikan
sedikit optimistis. Penilaian yang tidak bias hanya dari **gold_test** (bagian 9). Irisan "dilihat manusia" di gold_val
hanya 103 baris (24 positif), jadi selisih kecil di sana setara beberapa ulasan saja.

**Untuk ditinjau pemilik label (borderline campuran):** "jaringannya Uda lumayan bagus tp kuotanya hangus terus…" dan
"bagus, internet juga kuat… tapi sayangnya semakin di beli semakin mahal…" berlabel negative, sedangkan
"makin bagus tampilan n makin lengkap fitur2nya, tapi… poin jadi hangus semua" dan "Be honest, walaupun harga kuotanya
makin mahal, tapi pelayanannya best banget…" berlabel positive. Keempatnya sah menurut panduan "pilih yang dominan";
kalau ingin lebih konsisten, pertimbangkan aturan tambahan, misalnya "ada keluhan konkret yang belum selesai -> negative".

**Yang belum terselesaikan oleh baseline:** sarkasme, campuran, dan negasi/idiom (13 dari 25 error) butuh pemahaman
konteks. Ini alasan mencoba IndoBERT (bagian 7).

### Perbaikan setelah analisis (2026-10-27 s/d 28)

| Langkah | Yang berubah | Baseline GOLD_VAL semua / dilihat manusia |
|---|---|---|
| v1 | filter lama, threshold 0,5, label lama | 0,829 / 0,789 (n manusia = 103) |
| Filter B | NEG_HINTS diperluas (keluhan ringan/sopan) | 0,865 / 0,835 (label lama) |
| Relabel campuran | 116 ulasan campuran dilabeli ulang dengan aturan "ada keluhan konkret yang belum selesai -> negative"; 4 label berubah, 74 label Claude kini dilihat manusia | 0,860 / 0,818 (n manusia = 145) |
| **Filter C + threshold** | campuran bintang 4-5 (kata kontras + keluhan) dibuang dari training; threshold dipilih di VAL rating (0,60) | **0,893 / 0,866** |

Angka sebelum dan sesudah relabel tidak sepenuhnya sebanding karena labelnya berubah. Semua keputusan di atas dibuat di
gold_val, jadi angka gold_val optimistis; angka yang dilaporkan adalah gold_test (bagian 9).

In [16]:
# Eksperimen di gold_val: A = filter lama, B = NEG_HINTS diperluas, C = B + campuran bintang 4-5 dibuang (dipakai).
# Threshold kedua dipilih di VAL label rating (bukan di gold), lalu dicek di gold_val.
def threshold_table(model, val_df, gval_df, name):
    pos = list(model.classes_).index("positive")
    pv, pg = model.predict_proba(val_df)[:, pos], model.predict_proba(gval_df)[:, pos]
    yv, yg = val_df["label_rating"].to_numpy(), gval_df["label"].to_numpy()
    human = (gval_df["source"] != "claude_unchecked").to_numpy()
    lab = lambda p, t: np.where(p >= t, "positive", "negative")
    f1m = lambda y, p: f1_score(y, p, labels=LABELS, average="macro", zero_division=0)
    best = max(np.round(np.arange(0.30, 0.91, 0.05), 2), key=lambda t: f1m(yv, lab(pv, t)))
    return [{"varian": name, "threshold": t, "VAL_rating_F1": f1m(yv, lab(pv, t)),
             "GOLD_VAL_F1": f1m(yg, lab(pg, t)), "GOLD_VAL_manusia_F1": f1m(yg[human], lab(pg[human], t))}
            for t in sorted({0.5, float(best)})]

if gold_val_df is not None:
    rows = []
    for name, hints, contrast in [("A. filter lama", NEG_HINTS_V1, False), ("B. NEG_HINTS diperluas", NEG_HINTS, False)]:
        d_x = df.copy()
        d_x["conflict_flag"] = [conflict_flag(t, s, hints, contrast) for t, s in zip(d_x["text_light"], d_x["score"])]
        train_x, val_x = split_data(d_x, gold)
        rows += threshold_table(build_pipeline().fit(train_x, train_x["label_rating"]), val_x, gold_val_df, name)
    rows += threshold_table(pipe, val, gold_val_df, "C. B + campuran dibuang (dipakai)")
    exp = pd.DataFrame(rows)
    display(exp.round(3))

Dibuang dari data latih (positif berisik): 237 baris


Dibuang dari data latih (positif berisik): 343 baris


,varian,threshold,VAL_rating_F1,GOLD_VAL_F1,GOLD_VAL_manusia_F1
0,A. filter lama,0.50,0.874,0.837,0.793
1,A. filter lama,0.65,0.888,0.860,0.816
2,B. NEG_HINTS diperluas,0.50,0.883,0.860,0.818
3,B. NEG_HINTS diperluas,0.55,0.898,0.869,0.831
4,C. B + campuran dibuang (dipakai),0.50,0.887,0.869,0.831
5,C. B + campuran dibuang (dipakai),0.60,0.892,0.893,0.866


## 7. IndoBERT (fine-tune di Google Colab)
Training butuh GPU, jadi dilakukan di **`03_v2_indobert_colab.ipynb`**. Notebook ini hanya menyiapkan data dan menilai hasilnya.

1. Sel pertama di bawah membuat `data/bert_data.zip`: train & val (label rating, setelah filter positif berisik,
   sama persis dengan baseline) + **teks** gold_val dan gold_test **tanpa label**.
2. Buka `03_v2_indobert_colab.ipynb` di Colab (GPU), upload `bert_data.zip`, jalankan semua sel.
3. Ekstrak `indobert_results.zip` hasil unduhan ke `data/bert/`, lalu jalankan ulang sel kedua di bawah.

Input IndoBERT = `text_light` (tanpa stopword/stemming). Early stopping memakai macro-F1 di VAL label rating,
jadi gold tidak ikut memilih model. Angka yang dibandingkan dengan baseline = **GOLD_VAL**.

In [17]:
# -> src/export_bert.py
import hashlib, zipfile

def export_for_colab(train, val, gold_val, gold_test, out_dir=BERT_DIR, zip_path=DATA_DIR / "bert_data.zip"):
    out_dir.mkdir(parents=True, exist_ok=True)
    tr = pd.DataFrame({"text": train["text_light"], "label": train["label_rating"]})
    va = pd.DataFrame({"text": val["text_light"], "label": val["label_rating"]})
    gt = pd.concat([gold_val.assign(split="val"), gold_test.assign(split="test")])
    gt = pd.DataFrame({"reviewId": gt["reviewId"].astype(str), "split": gt["split"],
                       "text": gt["content"].map(light_clean)})          # sengaja TANPA label
    files = {"train.csv": tr, "val.csv": va, "gold_texts.csv": gt}
    digest, d_train, d_gold = hashlib.sha256(), hashlib.sha256(), hashlib.sha256()
    for name, d in files.items():
        csv = d.to_csv(index=False)
        (out_dir / name).write_text(csv, encoding="utf-8")
        digest.update(csv.encode())
        (d_gold if name == "gold_texts.csv" else d_train).update(csv.encode())
    manifest = {"data_hash": digest.hexdigest()[:16],
                "gold_hash": d_gold.hexdigest()[:16],    # menentukan sah/tidaknya prediksi (teks + split gold)
                "train_hash": d_train.hexdigest()[:16],  # hanya mencatat versi data training
                "versi": FILTER_VERSION, "n_train": len(tr), "n_val": len(va),
                "n_gold_val": int((gt["split"] == "val").sum()), "n_gold_test": int((gt["split"] == "test").sum()),
                "drop_noisy_positives": DROP_NOISY_POSITIVES, "seed": SEED}
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    hist = out_dir / "manifest_history.jsonl"
    known = [json.loads(l)["data_hash"] for l in hist.read_text().splitlines()] if hist.exists() else []
    if manifest["data_hash"] not in known:
        with open(hist, "a") as f:
            f.write(json.dumps(manifest, ensure_ascii=False) + "\n")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for name in [*files, "manifest.json"]:
            z.write(out_dir / name, arcname=f"bert/{name}")
    print(f"{zip_path} siap untuk Colab: {manifest}")
    return manifest

bert_manifest = export_for_colab(train, val, gold_val, gold_test) if gold is not None else None

data/bert_data.zip siap untuk Colab: {'data_hash': 'f5f396d397d46bc6', 'gold_hash': '42d0cc3823452ecb', 'train_hash': '7553b8add5d0fd32', 'versi': 'C: NEG_HINTS diperluas + campuran bintang 4-5 (kontras + keluhan) dibuang', 'n_train': 7474, 'n_val': 1320, 'n_gold_val': 248, 'n_gold_test': 248, 'drop_noisy_positives': True, 'seed': 42}


In [18]:
# -> src/evaluate.py (hasil IndoBERT dari Colab)
BERT_PREDS, BERT_INFO = BERT_DIR / "preds_indobert.csv", BERT_DIR / "run_info.json"
BERT_PREDS_VAL = BERT_DIR / "preds_val_rating.csv"   # probabilitas IndoBERT di VAL rating (untuk threshold)

def load_bert_preds(manifest):
    if not (BERT_PREDS.exists() and BERT_INFO.exists()):
        print("Belum ada hasil IndoBERT. Jalankan 03_v2_indobert_colab.ipynb, lalu ekstrak indobert_results.zip ke data/bert/.")
        return None, None
    info = json.loads(BERT_INFO.read_text())
    if info.get("smoke_test"):
        print("[PERINGATAN] run_info.json berasal dari SMOKE TEST (model kecil). Hasil ini tidak bermakna.")
    used = dict(info.get("data_manifest", {}))
    hist = BERT_DIR / "manifest_history.jsonl"
    if "gold_hash" not in used and hist.exists():          # run lama: cari gold_hash lewat riwayat ekspor
        for line in hist.read_text().splitlines():
            h = json.loads(line)
            if h["data_hash"] == used.get("data_hash"):
                used = h
    if manifest and used.get("gold_hash") != manifest["gold_hash"]:
        print(f"[DITOLAK] Prediksi IndoBERT dibuat untuk teks/split gold yang berbeda "
              f"({used.get('gold_hash')} vs sekarang {manifest['gold_hash']}). Jalankan ulang Colab.")
        return None, info
    info["train_versi"] = used.get("versi", "?")
    if manifest and used.get("train_hash") != manifest["train_hash"]:
        print(f"[CATATAN] IndoBERT ini dilatih dengan data training versi lain: '{used.get('versi', '?')}'. "
              f"Data sekarang: '{manifest['versi']}'. Perbandingan dengan baseline tidak setara sampai Colab dijalankan "
              f"ulang dengan bert_data.zip terbaru.")
    return pd.read_csv(BERT_PREDS, dtype={"reviewId": str}), info

bert_preds, bert_info = load_bert_preds(bert_manifest) if gold is not None else (None, None)
if bert_preds is not None:
    print({k: bert_info.get(k) for k in ["model_name", "model_version", "train_versi", "device", "best_epoch",
                                         "val_rating_macro_f1_best", "train_minutes", "latency_100_sec"]})
    p = bert_preds[bert_preds["split"] == "val"].set_index("reviewId")["proba_pos"]
    gold_val_df["proba_pos_bert"] = gold_val_df["reviewId"].astype(str).map(p)
    assert gold_val_df["proba_pos_bert"].notna().all(), "ada baris gold_val tanpa prediksi IndoBERT"
    BERT_THRESHOLD = 0.5
    if BERT_PREDS_VAL.exists() and bert_info.get("train_versi") == bert_manifest.get("versi"):
        pv = pd.read_csv(BERT_PREDS_VAL)["proba_pos"].to_numpy()
        if len(pv) == len(val):
            BERT_THRESHOLD = choose_threshold(val["label_rating"], pv)
        else:
            print(f"[PERINGATAN] preds_val_rating.csv berisi {len(pv)} baris, VAL {len(val)} -> threshold 0.5")
    print(f"Threshold IndoBERT: {BERT_THRESHOLD} "
          f"({'dipilih di VAL rating' if BERT_THRESHOLD != 0.5 or BERT_PREDS_VAL.exists() else 'default, tidak ada prediksi VAL'})")
    gold_val_df["pred_bert"] = to_label(gold_val_df["proba_pos_bert"], BERT_THRESHOLD)
    results_bert = eval_gold_split(gold_val_df, "pred_bert", "GOLD_VAL", "IndoBERT")
    ok_b, ok_i = gold_val_df["pred"] == gold_val_df["label"], gold_val_df["pred_bert"] == gold_val_df["label"]
    print("\nBaseline vs IndoBERT di gold_val (baris benar/salah):")
    print(pd.crosstab(ok_b.map({True: "benar", False: "salah"}), ok_i.map({True: "benar", False: "salah"}),
                      rownames=["baseline"], colnames=["IndoBERT"]))
    json.dump(results_bert, open(MODEL_DIR / "indobert_gold_val_report.json", "w"), indent=2, ensure_ascii=False)

{'model_name': 'indobenchmark/indobert-base-p1', 'model_version': 'indobert-v2-20261028-0837-f5f396', 'train_versi': 'C: NEG_HINTS diperluas + campuran bintang 4-5 (kontras + keluhan) dibuang', 'device': 'Tesla T4', 'best_epoch': 2.0, 'val_rating_macro_f1_best': 0.8971654006238632, 'train_minutes': 2.04, 'latency_100_sec': 0.157}
Threshold IndoBERT: 0.85 (dipilih di VAL rating)

=== GOLD_VAL IndoBERT (semua baris)  (n=248) ===
accuracy : 0.952
macro-F1 : 0.905   <- angka utama
  negative  P=0.99 R=0.95 F1=0.97 support=215
  positive  P=0.76 R=0.94 F1=0.84 support=33
                pred:negative  pred:positive
benar:negative            205             10
benar:positive              2             31
  irisan[sarkasme] n=12 acc=0.750 macroF1=0.621
  irisan[conflict_flag] n=120 acc=0.967 macroF1=0.658

=== GOLD_VAL IndoBERT (hanya dilihat manusia)  (n=145) ===
accuracy : 0.938
macro-F1 : 0.886   <- angka utama
  negative  P=0.98 R=0.94 F1=0.96 support=124
  positive  P=0.73 R=0.90 F1=0.81

In [19]:
# Apakah selisih baseline vs IndoBERT di gold_val nyata, atau bisa kebetulan? (+ ensemble rata-rata probabilitas)
from math import comb

def compare_models(gdf, n_boot=2000, seed=SEED):
    y = gdf["label"].to_numpy()
    preds = {"baseline": gdf["pred"].to_numpy(), "IndoBERT": gdf["pred_bert"].to_numpy(),
             "ensemble rata-rata": np.where((gdf["proba_pos"] + gdf["proba_pos_bert"]) / 2 >= 0.5, "positive", "negative")}
    human, sar = (gdf["source"] != "claude_unchecked").to_numpy(), (gdf["sarcasm"] == 1).to_numpy()
    f1m = lambda m, p: f1_score(y[m], p[m], labels=LABELS, average="macro", zero_division=0)
    allm = np.ones(len(y), bool)
    table = pd.DataFrame([{"model": k, "GOLD_VAL_F1": f1m(allm, p), "manusia_F1": f1m(human, p),
                           f"sarkasme_F1 (n={sar.sum()})": f1m(sar, p)} for k, p in preds.items()]).round(3)
    ob, oi = preds["baseline"] == y, preds["IndoBERT"] == y
    b, c = int((ob & ~oi).sum()), int((~ob & oi).sum())
    p_mcnemar = min(1.0, 2 * sum(comb(b + c, k) for k in range(min(b, c) + 1)) / 2 ** (b + c))
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        ix = rng.integers(0, len(y), len(y))
        diffs.append(f1_score(y[ix], preds["IndoBERT"][ix], average="macro")
                     - f1_score(y[ix], preds["baseline"][ix], average="macro"))
    diffs = np.array(diffs)
    return table, {"baseline_benar_BERT_salah": b, "baseline_salah_BERT_benar": c, "mcnemar_p": round(p_mcnemar, 3),
                   "boot_mean_selisih": round(float(diffs.mean()), 3),
                   "boot_CI95": [round(float(x), 3) for x in np.percentile(diffs, [2.5, 97.5])],
                   "P(selisih<=0)": round(float((diffs <= 0).mean()), 3)}

if bert_preds is not None:
    cmp_table, cmp_stats = compare_models(gold_val_df)
    display(cmp_table)
    print(cmp_stats)
    with pd.option_context("display.max_colwidth", 140):
        for title, m in [("Diperbaiki IndoBERT", (gold_val_df["pred"] != gold_val_df["label"]) & (gold_val_df["pred_bert"] == gold_val_df["label"])),
                         ("Dirusak IndoBERT", (gold_val_df["pred"] == gold_val_df["label"]) & (gold_val_df["pred_bert"] != gold_val_df["label"]))]:
            print(f"\n-- {title} ({int(m.sum())}) --")
            display(gold_val_df[m][["content", "label", "proba_pos", "proba_pos_bert", "sarcasm", "source"]])

,model,GOLD_VAL_F1,manusia_F1,sarkasme_F1 (n=12)
0,baseline,0.893,0.866,0.455
1,IndoBERT,0.905,0.886,0.621
2,ensemble rata-rata,0.900,0.879,0.556


{'baseline_benar_BERT_salah': 5, 'baseline_salah_BERT_benar': 6, 'mcnemar_p': 1.0, 'boot_mean_selisih': 0.011, 'boot_CI95': [-0.038, 0.062], 'P(selisih<=0)': 0.328}

-- Diperbaiki IndoBERT (6) --


,content,label,proba_pos,proba_pos_bert,sarcasm,source
33,pas,positive,0.252395,0.990235,1,human_blind
50,low respon.,negative,0.758822,0.013197,0,human_checked
51,aku cinta Telkomsel sampai mati,positive,0.419078,0.968886,0,human_checked
54,"paket cepat habis, signal kadang lemah. thanks 🙏😇🙏",negative,0.851658,0.063251,0,human_checked
95,saya top up 105.000 tapi kok cuma 85.000 kemana pulsa saya dasar MARUK,negative,0.715244,0.043896,0,human_checked
229,Apk my Telkomsel yg lama lebih baik dari yang sekarang. Kl sdh diperbaiki auto tambah bintang,negative,0.602439,0.225459,0,human_relabel



-- Dirusak IndoBERT (5) --


,content,label,proba_pos,proba_pos_bert,sarcasm,source
22,"makin bagus tampilan n makin lengkap fitur2nya, tapi sayangnya uda ga ada undian2 berhadiah lagi ya.. jadi mubazir poin yg sudah terkump...",negative,0.332870,0.961678,0,human_relabel
29,"baik, cuma terlalu boros makan paket",negative,0.242691,0.887497,1,human_relabel
31,Sudah bisa di buka aplikasi nya. Walaupun sangat lemod.,negative,0.395854,0.909827,1,human_blind
58,"bagus, internet juga kuat saat di daerah terpencil, tapi sayangnya semakin di beli semakin mahal, mungkin gak cocok di kantong saya",negative,0.588187,0.992334,0,human_relabel
67,lebih di tingkatkan kembali . saya keluarga memakai telkomsel.jaringan sering error 🙏,negative,0.583515,0.868715,0,human_checked


### Kesimpulan bagian 7 (final, 2026-10-28)

Tiga run IndoBERT (`indobenchmark/indobert-base-p1`, Tesla T4) tersimpan di `data/bert/run*/`:

| Run | Data training | Epoch | Threshold | GOLD_VAL semua | dilihat manusia |
|---|---|---:|---|---:|---:|
| 1 | filter B | 4 | 0,5 | 0,881* | 0,841* |
| 2 | filter C | 4 | 0,5 (tanpa prediksi VAL) | 0,881 | 0,850 |
| **3 (aktif)** | filter C | 2 | 0,85 (dipilih di VAL rating) | **0,905** | **0,886** |

\* dinilai dengan label setelah relabel.

Perbandingan setara (data C, threshold dipilih di VAL rating untuk keduanya):

| | Baseline C (t = 0,60) | IndoBERT run 3 (t = 0,85) |
|---|---:|---:|
| GOLD_VAL semua (n = 248) | 0,893 | 0,905 |
| GOLD_VAL dilihat manusia (n = 145) | 0,866 | 0,886 |
| Sarkasme (n = 12) | 0,455 | 0,621 |
| Ukuran model | 1,6 MB | ~500 MB |
| Inferensi 100 ulasan | 0,011 s (CPU Mac) | 2,19 s (CPU Mac) / 0,157 s (T4) |
| Training | detik (CPU) | 2,0 menit (GPU Colab) |

* Selisihnya **tidak bermakna**: IndoBERT memperbaiki 6 dan merusak 5 ulasan (McNemar p = 1,0); interval bootstrap
  selisih macro-F1 [-0,038; 0,062] mencakup 0.
* Kenaikan terbesar selama proses datang dari **data dan label** (filter positif berisik, aturan ulasan campuran,
  threshold), bukan dari arsitektur model.
* Sarkasme tetap titik lemah kedua model (irisan hanya 12 baris, jadi angkanya kasar).

**Keputusan untuk bagian 9 (2026-10-28):** model utama = **baseline TF-IDF + LR (filter C, threshold 0,60)**,
karena akurasinya setara secara statistik dengan IndoBERT, sementara ukuran, kecepatan CPU, dan biaya retrain jauh lebih
ringan. IndoBERT run 3 dilaporkan sebagai pembanding.

## 8. Coba prediksi

In [20]:
def predict_texts(texts, model=pipe):
    X = pd.DataFrame({"text_light": [light_clean(t) for t in texts],
                      "text_classic": [classic_clean(t) for t in texts]})
    out = pd.DataFrame(model.predict_proba(X), columns=list(model.classes_))
    out.insert(0, "label", to_label(out["positive"], THRESHOLD)); out.insert(0, "text", texts)
    return out

predict_texts(["bagus banget ya, bayar tapi kuota hilang terus",
               "tidak bagus, sering error",
               "lumayan lah, kadang lemot tapi masih bisa dipakai",
               "mantap, cepat dan lancar 👍"])

,text,label,negative,positive
0,"bagus banget ya, bayar tapi kuota hilang terus",negative,0.816823,0.183177
1,"tidak bagus, sering error",negative,0.948076,0.051924
2,"lumayan lah, kadang lemot tapi masih bisa dipakai",negative,0.860694,0.139306
3,"mantap, cepat dan lancar 👍",positive,0.000568,0.999432


## 9. Evaluasi akhir di gold_test (SEKALI)
Jalankan **hanya setelah** semua keputusan (filter, threshold, pilihan baseline vs IndoBERT) dibuat di gold_val.
Set `RUN_FINAL_TEST = True` di bagian 0, *Restart & Run All*, lalu kembalikan ke `False`.
Setiap pembukaan dicatat di `models/final_test_log.jsonl`, supaya jumlah pembukaan gold_test terlihat.

In [21]:
if RUN_FINAL_TEST and gold_test is not None:
    from datetime import datetime
    gold_test_df = prepare_gold_eval(gold_test, df)
    gold_test_df["pred"] = to_label(pipe.predict_proba(gold_test_df)[:, POS_IDX], THRESHOLD)
    final = eval_gold_split(gold_test_df, "pred", "GOLD_TEST", "baseline")
    if bert_preds is not None:
        p = bert_preds[bert_preds["split"] == "test"].set_index("reviewId")["proba_pos"]
        gold_test_df["pred_bert"] = to_label(gold_test_df["reviewId"].astype(str).map(p), BERT_THRESHOLD)
        final += eval_gold_split(gold_test_df, "pred_bert", "GOLD_TEST", "IndoBERT")
    json.dump(final, open(MODEL_DIR / "final_test_report.json", "w"), indent=2, ensure_ascii=False)
    with open(MODEL_DIR / "final_test_log.jsonl", "a") as f:
        f.write(json.dumps({"opened_at": datetime.now().isoformat(timespec="seconds"),
                            "threshold": {"baseline": THRESHOLD, "IndoBERT": globals().get("BERT_THRESHOLD")},
                            "filter_version": FILTER_VERSION,
                            "models": [r["name"] for r in final],
                            "macro_f1": {r["name"]: round(r["macro_f1"], 4) for r in final}}) + "\n")
    n_open = sum(1 for _ in open(MODEL_DIR / "final_test_log.jsonl"))
    print(f"\ngold_test sudah dibuka {n_open} kali (idealnya 1).")
else:
    print("gold_test belum dibuka (RUN_FINAL_TEST = False).")


=== GOLD_TEST baseline (semua baris)  (n=248) ===
accuracy : 0.948
macro-F1 : 0.898   <- angka utama
  negative  P=0.98 R=0.96 F1=0.97 support=212
  positive  P=0.79 R=0.86 F1=0.83 support=36
                pred:negative  pred:positive
benar:negative            204              8
benar:positive              5             31
  irisan[sarkasme] n=10 acc=0.800 macroF1=0.444
  irisan[conflict_flag] n=116 acc=0.940 macroF1=0.715

=== GOLD_TEST baseline (hanya dilihat manusia)  (n=135) ===
accuracy : 0.911
macro-F1 : 0.853   <- angka utama
  negative  P=0.95 R=0.94 F1=0.95 support=111
  positive  P=0.73 R=0.79 F1=0.76 support=24
                pred:negative  pred:positive
benar:negative            104              7
benar:positive              5             19
  irisan[sarkasme] n=10 acc=0.800 macroF1=0.444

=== GOLD_TEST IndoBERT (semua baris)  (n=248) ===
accuracy : 0.944
macro-F1 : 0.894   <- angka utama
  negative  P=0.98 R=0.95 F1=0.97 support=212
  positive  P=0.76 R=0.89 F1=0.82 su